# Rotating equipment and converter performance maps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/equinor/neqsim/blob/master/examples/notebooks/energy_networks/02_rotating_equipment_and_converter_maps.ipynb)

This notebook demonstrates motor-driven equipment, shaft coupling, VFD/part-load motor performance, and load-dependent generator/transformer/prime-mover efficiency.

**Implementation dependencies:** PRs #2608, #2609, and #2615.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = [Path(env_root).resolve()] if env_root else []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for candidate in candidates:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root. Set NEQSIM_PROJECT_ROOT.")

PROJECT_ROOT = find_neqsim_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))
from neqsim_dev_setup import neqsim_init, neqsim_classes

ns = neqsim_classes(neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=True))
JClass = ns.JClass
print("NeqSim workspace classes loaded")

In [ ]:
ElectricMotor = JClass("neqsim.process.equipment.energy.ElectricMotor")
Generator = JClass("neqsim.process.equipment.energy.Generator")
Transformer = JClass("neqsim.process.equipment.energy.Transformer")
PrimeMover = JClass("neqsim.process.equipment.energy.PrimeMover")
LoadEfficiencyCurve = JClass("neqsim.process.equipment.energy.LoadEfficiencyCurve")
ElectricMotorDriver = JClass("neqsim.process.equipment.compressor.driver.ElectricMotorDriver")
MechanicalShaft = JClass("neqsim.process.equipment.stream.MechanicalShaft")
EnergyBus = JClass("neqsim.process.equipment.stream.EnergyBus")
EnergyType = JClass("neqsim.process.equipment.stream.EnergyType")
EnergyPortMode = JClass("neqsim.process.equipment.stream.EnergyPortMode")
EnergyConverter = JClass("neqsim.process.equipment.energy.EnergyConverter")

## 2. Motor part-load and VFD capability

In [ ]:
motor_driver = ElectricMotorDriver(5000.0, 3000.0, 0.96)
motor_driver.setMinSpeedRatio(0.3)
motor_driver.setMaxSpeedRatio(1.2)
motor_driver.setHasVFD(True)

motor = ElectricMotor("5 MW compressor motor")
motor.setPerformanceModel(motor_driver)

shaft = MechanicalShaft("compressor shaft")
shaft.setSpeed(2400.0)
motor.connectEnergyStream(EnergyConverter.OUTPUT_PORT, shaft, EnergyPortMode.CALCULATED)

loads = [0.1, 0.25, 0.5, 0.75, 1.0]
motor_rows = []
available = motor.getAvailableShaftPower()
for fraction in loads:
    output = fraction * available
    eta = motor.getEfficiencyAtOutputPower(output)
    input_power = motor.getRequiredInputPowerForOutput(output)
    motor_rows.append({"load_fraction": fraction, "shaft_MW": output/1e6,
                       "electrical_MW": input_power/1e6, "efficiency": eta})

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
motor_df = pd.DataFrame(motor_rows)
motor_df

In [ ]:
ax = motor_df.plot(x="load_fraction", y="efficiency", marker="o", figsize=(8,4))
ax.set_xlabel("Load fraction (-)")
ax.set_ylabel("Efficiency (-)")
ax.set_title("Motor/VFD part-load efficiency")
ax.grid()
plt.tight_layout()
plt.show()

**Interpretation.** Fixed losses reduce efficiency at low load. A motor that is oversized for normal operation can consume materially more electricity than a constant-efficiency model predicts.

In [ ]:
ax = motor_df.plot(x="shaft_MW", y="electrical_MW", marker="o", figsize=(8,4), label="Electrical input")
ax.plot(motor_df["shaft_MW"], motor_df["shaft_MW"], marker="o", label="Shaft output")
ax.set_xlabel("Shaft output (MW)")
ax.set_ylabel("Power (MW)")
ax.set_title("Electrical input and useful shaft output")
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

**Interpretation.** The difference between electrical input and shaft output is conversion loss. Energy conservation is maintained at every operating point.

## 3. Generator and prime-mover efficiency curves

In [ ]:
curve = LoadEfficiencyCurve(
    [0.1, 0.25, 0.5, 0.75, 1.0],
    [0.70, 0.88, 0.94, 0.965, 0.97]
)

generator = Generator("main generator")
generator.setRatedOutputPower(10.0e6)
generator.setLoadEfficiencyCurve(curve)

prime = PrimeMover("gas turbine")
prime.setRatedOutputPower(12.0e6)
prime.setLoadEfficiencyCurve(LoadEfficiencyCurve(
    [0.1, 0.25, 0.5, 0.75, 1.0],
    [0.15, 0.25, 0.32, 0.36, 0.38]
))

mapped = []
for fraction in [0.1, 0.25, 0.5, 0.75, 1.0]:
    gen_output = fraction * 10.0e6
    gt_output = fraction * 12.0e6
    mapped.append({
        "load_fraction": fraction,
        "generator_efficiency": curve.getEfficiency(fraction),
        "prime_mover_efficiency": prime.getLoadEfficiencyCurve().getEfficiency(fraction),
        "generator_input_MW": generator.getRequiredInputPowerForOutput(gen_output)/1e6,
        "fuel_input_MW": prime.getRequiredInputPowerForOutput(gt_output)/1e6,
    })
mapped_df = pd.DataFrame(mapped)
mapped_df

In [ ]:
ax = mapped_df.plot(x="load_fraction", y=["generator_efficiency", "prime_mover_efficiency"],
                    marker="o", figsize=(8,4))
ax.set_xlabel("Load fraction (-)")
ax.set_ylabel("Efficiency (-)")
ax.set_title("Load-dependent converter efficiencies")
ax.grid()
plt.tight_layout()
plt.show()

**Interpretation.** Prime-mover efficiency typically changes more strongly with load than generator efficiency. This affects fuel use, emissions, and optimal train loading.

## Summary

Use detailed performance maps for concept selection, electrification studies, compressor-driver matching, and operating optimization. Retain constant-efficiency models for early screening when detailed data are unavailable.